> Adapted from *Dive into Deep Learning*, section [Gated Recurrent Units (GRU)](https://d2l.ai/chapter_recurrent-modern/gru.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 10.1: Long Short-Term Memory (LSTM)](10.01_lstm.ipynb) | Next: [Section 10.3: Deep Recurrent Neural Networks](10.03_deep-rnn.ipynb)

# Gated Recurrent Units (GRU)


As RNNs and particularly the LSTM architecture ([Section 10.1](https://d2l.ai/chapter_recurrent-modern/lstm.html#sec-lstm))
rapidly gained popularity during the 2010s,
a number of researchers began to experiment 
with simplified architectures in hopes 
of retaining the key idea of incorporating
an internal state and multiplicative gating mechanisms
but with the aim of speeding up computation.
The gated recurrent unit (GRU) (Cho et al., 2014) 
offered a streamlined version of the LSTM memory cell
that often achieves comparable performance
but with the advantage of being faster 
to compute (Chung et al., 2014).

In [ ]:
from divelet_ml import nn
import autograd.numpy as np

import divelet_ml as d2l

## Reset Gate and Update Gate

Here, the LSTM's three gates are replaced by two:
the *reset gate* and the *update gate*.
As with LSTMs, these gates are given sigmoid activations,
forcing their values to lie in the interval $(0, 1)$.
Intuitively, the reset gate controls how much of the previous state 
we might still want to remember.
Likewise, an update gate would allow us to control 
how much of the new state is just a copy of the old one.
[Fig. 10.2.1](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-1) illustrates the inputs for both
the reset and update gates in a GRU, 
given the input of the current time step
and the hidden state of the previous time step.
The outputs of the gates are given 
by two fully connected layers
with a sigmoid activation function.

Computing the reset gate and the update gate in a GRU model.

Mathematically, for a given time step $t$,
suppose that the input is a minibatch
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$ 
(number of examples $=n$; number of inputs $=d$)
and the hidden state of the previous time step 
is $\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$ 
(number of hidden units $=h$). 
Then the reset gate $\mathbf{R}_t \in \mathbb{R}^{n \times h}$ 
and update gate $\mathbf{Z}_t \in \mathbb{R}^{n \times h}$ are computed as follows:

$$
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xr}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hr}} + \mathbf{b}_\textrm{r}),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xz}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hz}} + \mathbf{b}_\textrm{z}),
\end{aligned} \tag{10.2.1}$$

where $\mathbf{W}_{\textrm{xr}}, \mathbf{W}_{\textrm{xz}} \in \mathbb{R}^{d \times h}$ 
and $\mathbf{W}_{\textrm{hr}}, \mathbf{W}_{\textrm{hz}} \in \mathbb{R}^{h \times h}$ 
are weight parameters and $\mathbf{b}_\textrm{r}, \mathbf{b}_\textrm{z} \in \mathbb{R}^{1 \times h}$ 
are bias parameters.


## Candidate Hidden State

Next, we integrate the reset gate $\mathbf{R}_t$ 
with the regular updating mechanism
in [(9.4.5)](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html#equation-rnn-h-with-state),
leading to the following
*candidate hidden state*
$\tilde{\mathbf{H}}_t \in \mathbb{R}^{n \times h}$ at time step $t$:

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{\textrm{hh}} + \mathbf{b}_\textrm{h}), \tag{10.2.2}$$

where $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}$ and $\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$
are weight parameters,
$\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$
is the bias,
and the symbol $\odot$ is the Hadamard (elementwise) product operator.
Here we use a tanh activation function.

The result is a *candidate*, since we still need 
to incorporate the action of the update gate.
Comparing with [(9.4.5)](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html#equation-rnn-h-with-state),
the influence of the previous states
can now be reduced with the
elementwise multiplication of
$\mathbf{R}_t$ and $\mathbf{H}_{t-1}$
in [(10.2.2)](https://d2l.ai/chapter_recurrent-modern/gru.html#equation-gru-tilde-H).
Whenever the entries in the reset gate $\mathbf{R}_t$ are close to 1, 
we recover a vanilla RNN such as that in [(9.4.5)](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html#equation-rnn-h-with-state).
For all entries of the reset gate $\mathbf{R}_t$ that are close to 0, 
the candidate hidden state is the result of an MLP with $\mathbf{X}_t$ as input. 
Any pre-existing hidden state is thus *reset* to defaults.

[Fig. 10.2.2](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-2) illustrates the computational flow after applying the reset gate.

Computing the candidate hidden state in a GRU model.


## Hidden State

Finally, we need to incorporate the effect of the update gate $\mathbf{Z}_t$.
This determines the extent to which the new hidden state $\mathbf{H}_t \in \mathbb{R}^{n \times h}$ 
matches the old state $\mathbf{H}_{t-1}$ compared with how much 
it resembles the new candidate state $\tilde{\mathbf{H}}_t$.
The update gate $\mathbf{Z}_t$ can be used for this purpose, 
simply by taking elementwise convex combinations 
of $\mathbf{H}_{t-1}$ and $\tilde{\mathbf{H}}_t$.
This leads to the final update equation for the GRU:

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1}  + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t. \tag{10.2.3}$$


Whenever the update gate $\mathbf{Z}_t$ is close to 1,
we simply retain the old state. 
In this case the information from $\mathbf{X}_t$ is ignored, 
effectively skipping time step $t$ in the dependency chain. 
By contrast, whenever $\mathbf{Z}_t$ is close to 0,
the new latent state $\mathbf{H}_t$ approaches the candidate latent state $\tilde{\mathbf{H}}_t$. 
[Fig. 10.2.3](https://d2l.ai/chapter_recurrent-modern/gru.html#fig-gru-3) shows the computational flow after the update gate is in action.

Computing the hidden state in a GRU model.


In summary, GRUs have the following two distinguishing features:

* Reset gates help capture short-term dependencies in sequences.
* Update gates help capture long-term dependencies in sequences.

## Implementation from Scratch

To gain a better understanding of the GRU model, let's implement it from scratch.

### Initializing Model Parameters

The first step is to initialize the model parameters.
We draw the weights from a Gaussian distribution
with standard deviation to be `sigma` and set the bias to 0. 
The hyperparameter `num_hiddens` defines the number of hidden units.
We instantiate all weights and biases relating to the update gate, 
the reset gate, and the candidate hidden state.

In [ ]:
class GRUScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()

        init_weight = lambda *shape: (np.random.randn(*shape) * sigma).astype(np.float32)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          np.zeros(num_hiddens, np.float32))

        self.W_xz, self.W_hz, self.b_z = triple()  # Update gate
        self.W_xr, self.W_hr, self.b_r = triple()  # Reset gate
        self.W_xh, self.W_hh, self.b_h = triple()  # Candidate hidden state

### Defining the Model

Now we are ready to define the GRU forward computation.
Its structure is the same as that of the basic RNN cell, 
except that the update equations are more complex.

In [ ]:
@d2l.add_to_class(GRUScratch)
def forward(self, inputs, H=None):
    if H is None:
        # Initial state with shape: (batch_size, num_hiddens)
        H = np.zeros((inputs.shape[1], self.num_hiddens), np.float32)
    outputs = []
    for X in inputs:
        Z = nn.sigmoid(np.matmul(X, self.W_xz) +
                       np.matmul(H, self.W_hz) + self.b_z)
        R = nn.sigmoid(np.matmul(X, self.W_xr) +
                       np.matmul(H, self.W_hr) + self.b_r)
        H_tilde = np.tanh(np.matmul(X, self.W_xh) +
                          np.matmul(R * H, self.W_hh) + self.b_h)
        H = Z * H + (1 - Z) * H_tilde
        outputs.append(H)
    return outputs, H

### Training

Training a language model on *The Time Machine* dataset
works in exactly the same manner as in [Section 9.5](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch).

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> In the browser, the two training cells below run fewer epochs than the book's 50: `d2l.browser_epochs(50, pyodide=20, xeus=50)` gives 50 in xeus-python and 20 in Pyodide, whose NumPy has no BLAS, so that the notebook finishes in a few minutes. The perplexity ends higher and the generated text is worse than the book's. Set `max_epochs=50` to train as long as the book does.

In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)
gru = GRUScratch(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLMScratch(gru, vocab_size=len(data.vocab), lr=4)
trainer = d2l.Trainer(max_epochs=d2l.browser_epochs(50, pyodide=20, xeus=50), gradient_clip_val=1)
trainer.fit(model, data)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 50 epochs, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train_ppl 5.839, val_ppl 7.053.

Training curves of the full run. Final values: train_ppl 5.839, val_ppl 7.053.

## Concise Implementation

In high-level APIs, we can directly instantiate a GRU model.
This encapsulates all the configuration detail that we made explicit above.

In [ ]:
class GRU(d2l.RNN):
    def __init__(self, num_inputs, num_hiddens):
        d2l.Module.__init__(self)
        self.save_hyperparameters()
        self.rnn = nn.GRU(num_inputs, num_hiddens)

The code is significantly faster in training as it uses compiled operators 
rather than Python.

In [ ]:
gru = GRU(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLM(gru, vocab_size=len(data.vocab), lr=4)
trainer.fit(model, data)

<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Full run, for comparison.** In JupyterLite, the cell above trains less than the book does (see the note before it). This figure is what it gives with 50 epochs, as in the book: computed with CPython, not in the browser, from a random start (seed 0), so a run of your own will differ in the details. Final values: train_ppl 4.923, val_ppl 6.665.

Training curves of the full run. Final values: train_ppl 4.923, val_ppl 6.665.

After training, we print out the perplexity on the training set
and the predicted sequence following the provided prefix.

In [ ]:
model.predict('it has', 20, data.vocab)

## Summary

Compared with LSTMs, GRUs achieve similar performance but tend to be lighter computationally.
Generally, compared with simple RNNs, gated RNNS, just like LSTMs and GRUs,
can better capture dependencies for sequences with large time step distances.
GRUs contain basic RNNs as their extreme case whenever the reset gate is switched on. 
They can also skip subsequences by turning on the update gate.


## Exercises

1. Assume that we only want to use the input at time step $t'$ to predict the output at time step $t > t'$. What are the best values for the reset and update gates for each time step?
1. Adjust the hyperparameters and analyze their influence on running time, perplexity, and the output sequence.
1. Compare runtime, perplexity, and the output strings for `rnn.RNN` and `rnn.GRU` implementations with each other.
1. What happens if you implement only parts of a GRU, e.g., with only a reset gate or only an update gate?

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/1056)

Previous: [Section 10.1: Long Short-Term Memory (LSTM)](10.01_lstm.ipynb) | Next: [Section 10.3: Deep Recurrent Neural Networks](10.03_deep-rnn.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Gated Recurrent Units (GRU)](https://d2l.ai/chapter_recurrent-modern/gru.html)
(source: [`chapter_recurrent-modern/gru.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_recurrent-modern/gru.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- Training for 50 epochs, twice, from scratch and with the high-level API, takes minutes in the browser, so `d2l.browser_epochs(50, pyodide=20, xeus=50)` gives 50 epochs outside JupyterLite but 50 in xeus-python and 20 in Pyodide. This is not in the book: in the browser, the curves show less training, and the perplexity is higher and the generated text worse than the book's. Set `max_epochs=50` to train as long as the book does.
- `nn` is divelet-ml's `divelet_ml.nn`: its `nn.GRU` is a NumPy stand-in for `torch.nn.GRU`, with the same inputs and outputs (a state of shape (number of layers, batch size, number of hidden units)) and PyTorch's gate equations, written as a Python loop over the time steps that autograd differentiates. So it is not faster than the from-scratch model here, unlike the book's PyTorch version. The `GRU` class of this section has one layer; the multilayer `d2l.GRU` of Section 10.3 is in divelet-ml.
- `d2l.sigmoid`, `d2l.tanh`, `d2l.matmul`, `d2l.zeros` and `d2l.randn` become `nn.sigmoid` and the functions of NumPy. `nn.Parameter` is dropped: arrays held as attributes are the model's parameters, so the from-scratch weights are `float32` arrays. `d2l.Trainer(..., num_gpus=1)`, `d2l.try_gpu()` and the `device` argument of `predict` are removed: a browser kernel has no GPU.
- After each training cell, a figure marked with the tablet shows what the book's full training gives, computed with CPython, not in the browser (made by `tools/full_run_figures.py`). It is not in the book.
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.